# 04 - Scikit-learn Modeli

Bu notebook, laboratuvar temel modelinin ayni mantigini `MLPClassifier` ile tekrar kurar.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.data_utils import RANDOM_SEED, prepare_datasets, set_global_seed
from src.metrics import evaluate_classification
from src.numpy_models import NumpyMLPClassifier, create_lab_initial_parameters
from src.sklearn_model import train_sklearn_baseline
from src.visualization import plot_confusion_matrix_figure


In [ ]:
set_global_seed(RANDOM_SEED)
DATA_PATH = PROJECT_ROOT / 'data' / 'heart_failure_clinical_records_dataset.csv'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
split = prepare_datasets(DATA_PATH, random_state=RANDOM_SEED)

shared_initial_parameters = create_lab_initial_parameters([split.X_train.shape[1], 8, 1], seed=RANDOM_SEED)

numpy_baseline = NumpyMLPClassifier([split.X_train.shape[1], 8, 1], learning_rate=0.05, epochs=400, seed=RANDOM_SEED, initialization='lab')
numpy_baseline.set_parameters(shared_initial_parameters)
numpy_baseline.fit(split.X_train, split.y_train, split.X_val, split.y_val)
numpy_test_pred = numpy_baseline.predict(split.X_test)
numpy_metrics = evaluate_classification(split.y_test, numpy_test_pred, numpy_baseline.predict_proba(split.X_test))

sklearn_model, sklearn_history = train_sklearn_baseline(
    split.X_train,
    split.y_train,
    X_val=split.X_val,
    y_val=split.y_val,
    max_iter=400,
    learning_rate_init=0.05,
    initial_parameters=shared_initial_parameters,
)
sklearn_test_pred = sklearn_model.predict(split.X_test)
sklearn_metrics = evaluate_classification(split.y_test, sklearn_test_pred, sklearn_model.predict_proba(split.X_test)[:, 1])

comparison_df = pd.DataFrame([
    {'model': 'NumPy baseline', 'accuracy': numpy_metrics['accuracy'], 'precision': numpy_metrics['precision'], 'recall': numpy_metrics['recall'], 'f1_score': numpy_metrics['f1_score']},
    {'model': 'Scikit-learn baseline', 'accuracy': sklearn_metrics['accuracy'], 'precision': sklearn_metrics['precision'], 'recall': sklearn_metrics['recall'], 'f1_score': sklearn_metrics['f1_score']},
])
comparison_df

In [ ]:
plot_confusion_matrix_figure(split.y_test, sklearn_test_pred, FIGURE_DIR / 'sklearn_confusion_matrix.png', 'Scikit-learn Baseline Confusion Matrix')
print(sklearn_metrics['classification_report'])

Kisa yorum: Scikit-learn modeli, ayni split, ayni 8 noronlu tek gizli katmanli mimari, `tanh` aktivasyonu, `SGD` cozucusu ve ortak laboratuvar tipi baslangic agirliklari ile calistirilmistir. Bu nedenle burada yalnizca ayni veri bolmesi degil, temel mimarinin cekirdek kurulumu da referans model ile hizalanmistir.